In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import wandb
import joblib
import os
from sklearn.preprocessing import LabelEncoder
!pip install wandb -q

import wandb
import os
!pip install pytorch-forecasting pandas numpy torch matplotlib

In [3]:
%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()
print(df_test_full.head())

Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [6]:
api_key = "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"

if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sandro/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
import wandb
import joblib
import os

api = wandb.Api()

# Define the run path components
entity = "slomi23-free-university-of-tbilisi-"
project = "walmart-sales-forecasting"
run_id = "n61o8psf"

# Fetch the specific run
run = api.run(f"{entity}/{project}/{run_id}")

# Find the model artifact in this run
model_artifact = None
for artifact in run.logged_artifacts():
    if artifact.type == 'model':
        model_artifact = artifact
        break

if model_artifact is None:
    raise Exception("No model artifact found in this run.")

# Download the artifact to a local directory
model_dir = model_artifact.download("model")

# List files to find the model file
files = os.listdir(model_dir)
print(f"Files downloaded: {files}")

# Load the first .joblib or .pkl file found
model_path = None
for f in files:
    if f.endswith('.joblib') or f.endswith('.pkl'):
        model_path = os.path.join(model_dir, f)
        break

if not model_path:
    raise Exception("No .joblib or .pkl file found in the artifact.")

best_model = joblib.load(model_path)
print("Model loaded successfully!")


wandb:   1 of 1 files downloaded.  


Files downloaded: ['model.joblib']
Model loaded successfully!


In [16]:
print(df_test_full.columns.tolist())


['Store', 'Dept', 'Date', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size']


In [28]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. PREPARE TEST DATA & ENGINEER FEATURES
# ---------------------------------------------------------
df_test_full['Date'] = pd.to_datetime(df_test_full['Date'])
df_test_full = df_test_full.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

# Extract Time Features
df_test_full['Year'] = df_test_full['Date'].dt.year
df_test_full['Month'] = df_test_full['Date'].dt.month
df_test_full['DayOfWeek'] = df_test_full['Date'].dt.dayofweek
df_test_full['WeekOfYear'] = df_test_full['Date'].dt.isocalendar().week.astype(int)
df_test_full['IsHoliday_int'] = df_test_full['IsHoliday'].astype(int)

# Encode Type (Note: lowercase 'encoded' based on error message)
type_mapping = {'A': 0, 'B': 1, 'C': 2}
df_test_full['Type_encoded'] = df_test_full['Type'].map(type_mapping)

# Convert IsHoliday to int if it's boolean
if 'IsHoliday' in df_test_full.columns:
    df_test_full['IsHoliday_int'] = df_test_full['IsHoliday'].astype(int)

# Fill Markdowns
for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
    df_test_full[col] = df_test_full[col].fillna(0)

# Get unique Store-Dept pairs
store_depts = df_test_full[['Store', 'Dept']].drop_duplicates()
all_predictions = []

print(f"Processing {len(store_depts)} Store-Dept combinations...")

# ---------------------------------------------------------
# 2. RECURSIVE PREDICTION LOOP WITH ALL LAGS
# ---------------------------------------------------------
for idx, row in store_depts.iterrows():
    store_id = row['Store']
    dept_id = row['Dept']
    
    mask = (df_test_full['Store'] == store_id) & (df_test_full['Dept'] == dept_id)
    subset = df_test_full[mask].copy()
    
    preds = []
    
    # Get historical sales from TRAINING SET
    train_mask = (df_train_full['Store'] == store_id) & (df_train_full['Dept'] == dept_id)
    hist_sales = df_train_full[train_mask]['Weekly_Sales'].values
    
    # We need enough history for max lag (52 weeks) + rolling windows
    # If hist_sales is too short, pad with zeros or mean
    min_history_needed = 52 
    if len(hist_sales) < min_history_needed:
        padding = np.zeros(min_history_needed - len(hist_sales))
        hist_sales = np.concatenate((padding, hist_sales))
        
    # Pointer to the most recent known sale in hist_sales
    curr_idx = len(hist_sales) - 1
    
    for i, (_, test_row) in enumerate(subset.iterrows()):
        feat_dict = {}
        
        # Static/Time features
        static_cols = [
            'IsHoliday', 'Temperature', 'Fuel_Price', 
            'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 
            'CPI', 'Unemployment', 'Size', 
            'Year', 'Month', 'DayOfWeek', 'WeekOfYear', 'Type_encoded', 'IsHoliday_int'
        ]
        
        for col in static_cols:
            if col in test_row.index:
                feat_dict[col] = [test_row[col]]
            else:
                feat_dict[col] = [0]
            
        # Dynamic Lag Features
        # Helper function to get value at offset k from current index
        def get_lag(k):
            target_idx = curr_idx - k
            if target_idx >= 0:
                return hist_sales[target_idx]
            else:
                # Fallback if we run out of history (shouldn't happen with padding)
                return 0 

        # Add all required lags
        feat_dict['sales_lag_1'] = [get_lag(1)]
        feat_dict['sales_lag_2'] = [get_lag(2)]
        feat_dict['sales_lag_4'] = [get_lag(4)]
        feat_dict['sales_lag_52'] = [get_lag(52)]
        
        # Rolling Statistics
        # roll_mean_4: Mean of last 4 weeks (indices curr-1, curr-2, curr-3, curr-4)
        window_4 = [hist_sales[curr_idx-j] for j in range(1, 5) if curr_idx-j >= 0]
        feat_dict['roll_mean_4'] = [np.mean(window_4) if window_4 else 0]
        feat_dict['roll_std_4'] = [np.std(window_4) if window_4 else 0]
        
        # roll_mean_8: Mean of last 8 weeks
        window_8 = [hist_sales[curr_idx-j] for j in range(1, 9) if curr_idx-j >= 0]
        feat_dict['roll_mean_8'] = [np.mean(window_8) if window_8 else 0]
        #feat_dict['IsHoliday_int'] = [feat_dict['IsHoliday'].astype(int).iloc[i]]

        expected_cols = [
            'IsHoliday', 'Temperature', 'Fuel_Price', 
            'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 
            'CPI', 'Unemployment', 'Size', 
            'sales_lag_1', 'sales_lag_2', 'sales_lag_4', 'sales_lag_52', 
            'roll_mean_4', 'roll_std_4', 'roll_mean_8', 
            'Year', 'Month', 'DayOfWeek', 'WeekOfYear', 
            'Type_encoded', 'IsHoliday_int'
        ]
        
        # Create DataFrame with columns in the correct order
        X_single = pd.DataFrame(feat_dict)[expected_cols]
        
        # Predict
        pred = best_model.predict(X_single)
        pred = max(0, pred)
        
        preds.append(pred)
        
        # Update history: Append prediction to hist_sales so next iteration can use it
        hist_sales = np.append(hist_sales, pred)
        curr_idx += 1

    for i, (_, test_row) in enumerate(subset.iterrows()):
        all_predictions.append({
            'Id': f"{test_row['Store']}_{test_row['Dept']}_{test_row['Date'].strftime('%Y-%m-%d')}",
            'Weekly_Sales': preds[i]
        })
    
    if idx % 100 == 0:
        print(f"... Processed {idx}/{len(store_depts)}")

# ---------------------------------------------------------
# 3. SAVE SUBMISSION
# ---------------------------------------------------------
submission_df = pd.DataFrame(all_predictions)
submission_df.to_csv('submission_xgboost_recursive.csv', index=False)
print("Submission saved as submission_xgboost_recursive.csv")


Processing 3169 Store-Dept combinations...
... Processed 0/3169
... Processed 2900/3169
... Processed 4300/3169
... Processed 9300/3169
... Processed 12800/3169
... Processed 13300/3169
... Processed 21200/3169
... Processed 23600/3169
... Processed 27300/3169
... Processed 28700/3169
... Processed 29900/3169
... Processed 33700/3169
... Processed 37100/3169
... Processed 43800/3169
... Processed 47500/3169
... Processed 49200/3169
... Processed 51400/3169
... Processed 52700/3169
... Processed 56100/3169
... Processed 62500/3169
... Processed 67900/3169
... Processed 68200/3169
... Processed 71300/3169
... Processed 82400/3169
... Processed 86000/3169
... Processed 94700/3169
... Processed 101400/3169
Submission saved as submission_xgboost_recursive.csv


In [29]:
import pandas as pd
import ast

# 1. Load the bad submission file
sub_df = pd.read_csv('submission_xgboost_recursive.csv')

# 2. Clean the Weekly_Sales column
def clean_value(val):
    try:
        # If it's a string like "[123.45]", parse it as a literal expression
        if isinstance(val, str) and val.startswith('['):
            parsed = ast.literal_eval(val)
            return float(parsed) if isinstance(parsed, list) else float(parsed)
        # If it's already a number, return it
        return float(val)
    except:
        return 0.0

# Apply cleaning
sub_df['Weekly_Sales'] = sub_df['Weekly_Sales'].apply(clean_value)

# 3. Save the cleaned file
sub_df.to_csv('submission_cleaned.csv', index=False)
print("Cleaned submission saved as submission_cleaned.csv")


Cleaned submission saved as submission_cleaned.csv
